# Transformer Based Character Level Small Language Model

This is a character level generative model built on Transformers, for sequence modeling. It looks at the context of input sequence and predicts the next likely character in the sequence.

Here character level small language model will use the transformer architecture to under go pre-training to model all the patterns present in the input data.


What is GPT and how it compares to GPT1?

GPT - Generative Pre-Trained Transformer - A Large language Model. is a production grade system with pretraining and fine-tuning that is complex.

- Transformers is the neural network that under the hood, that models sequence of words in GPT. It can look at the entire sequence and train in parallel. This character level model uses same Transformer architecture.
- Tokenization: It the process of segmenting input to atomic units/elements. Set of atomic units that the model will ever see or emit is called vocabulary. The vocabulary is numerically indexed. All inputs to the model are sequences of tokens, represented by numerical indices of the vocabulary. More atomic tokens give longer sequences but smaller vocabulary. Sequence lengths and vocabulary size are trade offs between each other. Different models use different schemas like GPT uses Byte Pair encoding. Google uses sentence piece (subword) tokenizer. For this model the units/schema are character level
- Generative - models joint probability of words - predicts next most likely token in the sequence. in a GPT model it token are at sub-word level. In the model trained here token is at character leve.
- Pretrained - learns the general representation of the world first - which can then be fine tuned for specific jobs like translation. GPT1 was Pre-trained on 4.5K MB of high quality webtext just to learn strucutre of English works—grammar, context etc. In this notebook, model is a very simple one, trained at a small scale of 1MB Shakespear works text and learns the structure of Shakespear writing
- GPT1 Supervised Fine-tuning: Once it "understood" English, it was given small, labeled datasets for specific tasks like Natural Language Inference (NLI)— in deciding if one sentence logically follows another. In this notebook, the character level small language model is not fine tuned
- The character level model is a decoder only model. GPT is also a decoder only model

References:
Attention is all you need - proposed Transformer model https://arxiv.org/abs/1706.03762


Dropouts: https://www.jmlr.org/papers/volume15/srivastava14a/srivastava14a.pdf?utm_content=buffer79b4

LayerNorm: https://arxiv.org/abs/1607.06450

Andrej Karapathy https://www.youtube.com/watch?v=kCc8FmEb1nY&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=8


In [21]:
import torch
torch.cuda.is_available()

True

### Notes on Self Attention
##### What problem does self attention solve?
Each token in sequence needs to gather information from its past in a data dependent way
That is take information from each past token based on how important they are (weights) - in a data dependent way
1. Create wei = matrix that hold interaction / affinity strengths]

##### How to estimate how imporant they are (wei)?

2. Calculate What information does each token in the past, provide to the current token

##### How can this be done in self attention?

Let each token emit - info that it provides in a $'key'$ and info it seeks in a $'query'$
The current token can then search among tokens in the past that is interested in by:
matching its query with the keys of tokens in the past
Match is done with dot product of query this token with keys of all tokens
; current query emit a query (seeking who matches it for that info of interest that it seeks)
Normalize dot product, to change it to proportions (sum to 1) - done with $softmax$

Variance of dot product increases d times where d is length of vector - scale by $sqrt(d)$ to get unit stdard deviation (std gaussain). This scaling avoids pushing the softmax function into regions where gradients are extremely small (vanishing gradients)
This gives the affinities; now dot product with the $'values'$ of each token
This is weighted sum of values of past tokens: weigth based on how interesting they are to the data in the input $X$ (data includes position information as well)

$Q = X \cdot W_q^T $ where $W_q \ shape = embed\_size, head\_size; X \ shape = context\_size, embed\_size $

$K = X \cdot W_k^T $ where $W_k \ shape = embed\_size, head\_size; X \ shape = context\_size, embed\_size $

$V = X \cdot W_v^T $ where $W_v \ shape = embed\_size, head\_size; X \ shape = context\_size, embed\_size $

Thus K, Q, V are data dependent on X. They are each a square matrix of context size. Let $d_k$ be head_size of a single head of attention. For a single token.
$ self \ \ attention = softmax(\frac{q \cdot K^T}{\sqrt d_k}) \cdot V$

Attention matrix for all tokens in the sequence where each token issues it own query - $q$
$$ self \ \ attention = softmax(\frac{Q \cdot K^T}{\sqrt d_k}) \cdot V$$




In [22]:
import torch # Pytorch: https://pytorch.org
import torch.nn as nn
import numpy as np
from torch.nn import functional as F

# hyperparameters
torch.manual_seed(1337) # for reproducibility
batch_size = 64 # 32 # number of independent sequences processed in parallel
block_size = 256 # 8 # maximum context length for predictions
max_iters = 5000 # max iterations increased as learning rate is reduced
eval_interval = 500
learning_rate = 3e-4 # le-2 for simple model but attention needs lower learning rate
device = 'cuda' if torch.cuda.is_available() else 'cpu' #use cuda / GPU if available
eval_iters = 200
vocab_size = 0 # will be initialized after reading data
n_embd = 384 # 32
n_head = 6
head_size = n_embd // n_head
n_layers = 6 # number of transformer blocks
dropout = 0.2
#------------


### Notes on Dropouts
Add dropout for regularization. Dropout rate $d$ and retention rate $p = 1-d$
Activations are switched off randomly at the rate of $d$ during training (forward and backward pass). This prevents the network from relying on all the input nodes to be present and prevents overfitting. In other words in each forward and backward pass diferent combinations of neurons are switched on creating ensemble of subnetworks; they are trained over many many epochs. During inference the full network is puttogether (like combining ensembles)

Weights are updated to only the participating activations. They will be sligthly higher as they compensate for the activations that are off. During inference the dropout is 0% (switched off), we scale the input activation $h$ by retention rate $(h*p)@ W$. This is equivalent to averaging the results of $h_1 @ W, h_2 @ W $ and so on n-ensembles of network, that are possible with $d %$ dropout combinations.

Another way is inverse dropout or inverted dropout (used in Pytorch and other libraries) where during training calculate input activation, scale it high so that the weights are learned to be lower
$ h = h/p$ during training. During inference then $h @ W$ can be used simply


Reference https://www.jmlr.org/papers/volume15/srivastava14a/srivastava14a.pdf?utm_content=buffer79b4


In [23]:
# Character level decoder only model that predicts next char based on input char
class CharLevelLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # each token directly reads off for the next token from a lookup table
        # since each row is vocab size it serves as the logits
        # parameters are moved to GPU if model is moved to device
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) # to encode token
        self.position_embedding_table = nn.Embedding(block_size, n_embd) # to encode position of token
        self.blocks = nn.Sequential(*[TransformerBlock(n_embd, n_head=n_head) for _ in range(n_layers)])
        self.layer_norm = nn.LayerNorm(n_embd) #final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size) # output head


    def forward(self, idx, target=None):
        #idx and targets are both (B, T) tensor of integers
        B, T = idx.shape
        x_emb = self.token_embedding_table(idx) # x_emb are B, T, C
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # T, C
        x = x_emb + pos_emb # shape T, C broadcast to B
        # placing attention and feedforward inside block
        x = self.blocks(x)
        x = self.layer_norm(x)
        logits = self.lm_head(x) # (B, T, vocab_size)
        B, T, vocab_size = logits.shape #(B,T,vocab_size) batch, block_size, embedding
        if target is None:
            loss = None
        else:
            # Cross Entropy - If dimensions > 2, pytorch expects embeddings at dim=1 --> B,vocab,T or B,vocab
            loss = F.cross_entropy(logits.view(B*T, vocab_size), target.view(B*T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        # get the prediction for T+1
        # pick the C for that last prediction, convert to probabilities and sample from it
        for _ in range(max_new_tokens):
            # get the predictions
            # crop idx to the last block_size tokens - it grows as it loops
            idx_cond = idx[:, -block_size:]
            # self(idx) calls forward function, but Y is not passed update forward to take Y optionally
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C) # Get the C for prediction of last T (timestep)
            # apply softmax to get the probabilities
            probs = F.softmax(logits, dim=-1) # (B, C) # Convert logits to probabilities
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) #(B, 1) # Sample from prob distribution
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

class SingleHeadAttention(nn.Module):
    "one head self-attention"
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout) # dropout after softmax to regularize affinity matrix

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x) # B, T, head_size
        q = self.query(x) # B, T, head_size
        head_size = q.shape[-1]
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * head_size**-0.5 # (B, T, H) @ (B, H, T) ---> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B, T, head_size)
        out = wei @ v # (B, T, T) @ (B, T, head_size) ---> (B, T, head_size)
        return out

class MultiHeadAttentionWithProjection(nn.Module):
    # multiple heads of self attention in parallel
    def __init__(self, num_heads, head_size):
        # n_embd: embedding dimension
        # n_head: number of heads
        super().__init__()
        self.heads = nn.ModuleList([SingleHeadAttention(head_size) for _ in range(num_heads)])
        #create projections of output to x sizes, to allow for additions of skip connections
        self.project = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # each head is run in parallel with the same input
        head_outputs = [h(x) for h in self.heads]
        out = torch.cat(head_outputs, dim=-1)
        out = self.project(out)
        out = self.dropout(out) # dropout before joining the residual connection
        # concatenate output from each head in the last dimension (head_size)
        # output size after concatenation is num_heads * head_size
        return out

# going directly from the multihead output to the language model head does not give the model a chance
# to think through or consolidate the outputs from multiple heads
# W = n_embd X n_embd example is a single set that is broadcast per token and applied in parallel
# Each token thinks through the data individually
class FeedForwardWithProjection(nn.Module):
    # a simple linear layer followed by a non-linearity
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd*4),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd), #projection layer
            nn.Dropout(dropout), # at the point of joining back the residual connection
        )

    def forward(self, x):
        out = self.net(x)
        return out

class TransformerBlock(nn.Module):
    # Transformer block, communication followed by computation
    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension
        # n_head: number of heads
        super().__init__()
        head_size = n_embd // n_head
        self.self_attention = MultiHeadAttentionWithProjection(n_head, head_size)
        self.ffwd = FeedForwardWithProjection(n_embd)
        self.layernorm1 = nn.LayerNorm(n_embd) # Layer norm from Pytorch # MyLayerNorm1d(n_embd)
        self.layernorm2 = nn.LayerNorm(n_embd) # Second LayerNorm after FFNN # MyLayerNorm1d(n_embd)

    def forward(self, x):
        x = x + self.self_attention(self.layernorm1(x)) # prenorm/layernorm -> self-attention -> residual/skip connection
        x = x + self.ffwd(self.layernorm2(x)) # prenorm / layernorm -> ffnn -> residual/skip connection
        return x


### Notes on Layer Normalization

The output z is the sum of many weighted inputs: $z=w1​x1​+w2​x2​+⋯+wn​xn​$ If each term $wi​xi​$ has a variance of 1, and you have $n$ inputs, the variance of the sum $z$ becomes $n$. We do not want the variance of this distribution of preactivation data be too broad (wide standard deviation) - else tanh gets saturated. If it is too narrow then tanh does not vary it much (non-linearity not fully utilized).
What we need is inputs to the activations need to be roughly Gaussian $(N(0,1))$- unit deviation and mean zero - so normalize it before passing it through the activation. Standardization (z-score normalization) is perfectly differentiable. Within the sample, normalize the inputs across its feature or embedding dimension, that go to the activation layer.

Reference https://arxiv.org/abs/1607.06450

Standard deviation/variance is measure of spread of the data

Standardization (z-score normalization) or Layer Normalizaton is
$$\frac{x - \mu_{L}}{\sigma_{L}} \ \ where \ \ \mu_{L} = \frac{1}{m}\sum_{i=1}^m x_i \ \ is \ \ row fetures \ \ mean \ \ and \ \ \sigma_{L} = \frac{1}{m}\sum_{i=1}^m (x_i - \mu_L)^2 \ \ is \ \ roe feature \ \ variance $$

This normalization fixes the data at the centre and scales it down; in other words it does not let the distribution move around the axis (to shift on the axis to be more trigger happy or less trigger happy) and change shape - to be diffused or sharp - as the neural networks wants it to, based on its learning from backprop. For this Scale and Shift are layers are introduced, which can be learned during training. In essence the scale and shift are pieced together to the normal distribution so that network can explicitly assign scale to spread the data if it wants to move parts of the data to where activation saturates or shift the data or both to achieve the non-linearity it needs. If the optimization learns that it does not need this scale and shift - it will remain standard gaussian and flow well through the activations with almost linear (minimal non-linear) transformation. This way the network is given the choice to move away from standard Gaussian if it helps it to learn better or avoid it if needed.

$$ \hat{x_i} = \frac{x_i - \mu_l}{\sqrt{\sigma_L^2 + \epsilon}} \ \ and \ \ y_i = \gamma\hat{x_i} + \beta \ \ where \ \ y_i \ \ is \ \ LN_{\gamma, \beta}(x_i)$$

If the network learns $\gamma = \sigma \ \ and \ \ \beta = \mu$ batch norm cancels itself and if it learns $\gamma = 1 \ \ and \ \ \beta = 0$ batch norm maintains it as standard gaussian  

For a deep network it will be intractable to manage the weight to keep the activations roughly gaussian
without batch normalization. It is normal to append batch normalization after every linear layer and convolution layer to control the scale of their activations at every point. It significantly stabilizes the training.

In [24]:
# Layer Norm Implemetation listed for illustration. In the model nn.LayerNorm Pytorch's implementation is used
class MyLayerNorm1d(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # parameters (trained with backprop)
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def forward(self, x):
         #def __call__(self, x):
        # calculate the forward pass
        xmean = x.mean(1, keepdim=True) # mean across the sample's dimensions (features)
        xvar = x.var(1, keepdim=True, correction=0) #, unbiased=True) # variance across the sample's features (dimension)
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
        self.out = self.gamma + xhat + self.beta
        return self.out



### Notes on Tokenization
Tokenization: segment the input to atomic units/elements. For this model the units are character level
Different models use different schemas like Byte Pair encoding,sentence piece (subword) tokenizer
Set of atomic units that the model will ever see or emit is called vocabulary
The vocabulary is numerically indexed.
All inputs to the model are sequences of tokens, represented by numerical indices of the vocabulary.
More atomic tokens give longer sequences but smaller vocabulary
Here we implement a simple tokenizer.
Character vocabulary is 65, sequences are long, but sub-word tokens in tiktoken: 5K give short sequence
Sequence lengths are vocabulary size are trade offs between each other

In [25]:
# read input file

with open('/content/tiny_shake/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()


# Tokenization: segment the input to atomic units/elements. For this model the units are character level
# All inputs to the model are sequences of tokens, represented by numerical indices of the vocabulary.
chars_input = sorted(set(text)) # create character level vocabulary
vocab_size = len(chars_input)
# map the vocabulary to integers
stoi, itos = {}, {}
for i, c in enumerate(chars_input):
    stoi[c] = i
    itos[i] = c
encode = lambda str_input: [stoi[c] for c in str_input] # encoder: input-string, output-sequence of int
decode = lambda ix_input: "".join([itos[i] for i in ix_input]) # decoder: input-sequence of int, output-string

# Encoding the entire text dataset and storing into a torch.Tensor
data = torch.tensor(encode(text), dtype=torch.long)
# splitting data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest validation
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate small batch of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(low = 0, high = len(data)-block_size, size=(batch_size, ))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device) # move data to cuda if it exists
    return x, y

# tell pytorch that backward will never be called on this function's computation graph
# pytorch does not store the intermediate steps in anticipation of backward pass, be memory efficient
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval() # change model to evaluation mode. In layers BatchNorm, Dropout etc training is set to False
    for split in ['train', 'val']: # evaluate over both splits
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters): # evaluate over a few batches
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean() # use the mean over few batches to get stable estimates
    model.train() # set the mode back to training = True in the layers
    return out



### Training the characterlevel Transformer Model

In [26]:
# Training the characterlevel Transformer Model
# Instantiate the model
model = CharLevelLanguageModel() # vocab_size is a global variable
m = model.to(device) # move model parameters essentially to cuda if it exists
print(f'number of parameters in the model {sum([np.prod(p.size()) for p in m.parameters()])})
# optimization typically is done with SGD. Here we use AdamW optimizer - lr is typically 1e-4 or 1e-3
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate) # for small networks the lr can take higer values
# train for max_iters epochs
for iter in range(max_iters):
    # every once in a while evaluate loss a few batches of training and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of training data
    xb, yb = get_batch('train')
    # evaluate the loss
    logits, loss = m(xb, yb) # forward pass
    optimizer.zero_grad(set_to_none=True) # flush the gradients
    loss.backward() # backpropagation
    optimizer.step() # update parameters by -lr*gradients

    # break # to trial run

# to generate a sample start token is idx=0 is '\n'
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("generated text: ", decode(m.generate(context, max_new_tokens=100)[0].tolist()))

10788929
step 0: train loss 4.2846, val loss 4.2820
step 500: train loss 1.8865, val loss 2.0022
step 1000: train loss 1.5361, val loss 1.7221
step 1500: train loss 1.3952, val loss 1.6038
step 2000: train loss 1.3081, val loss 1.5497
step 2500: train loss 1.2525, val loss 1.5164
step 3000: train loss 1.2010, val loss 1.4919
step 3500: train loss 1.1589, val loss 1.4809
step 4000: train loss 1.1225, val loss 1.4788
step 4500: train loss 1.0845, val loss 1.4730
generated text:  
But with price of this nightity-night
Shakes unstrumpet shall compell'd flood at their hand:
Commend


In [28]:
print(f'number of parameters in the model {sum([np.prod(p.size()) for p in m.parameters()])}')

number of parameters in the model 10788929


In [30]:
# to generate a sample start token is idx=0 is '\n'
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("generated text: ", decode(m.generate(context, max_new_tokens=1000)[0].tolist()))

generated text:  
ere old destard of rien: come to-day's blest your
own then hence of i a great dead. I'll respect off, I
till I cle you gone: if it bar out, in good came to
more banished, which I pandon no petitic with
Castle, were he note the courts. This come to heaven, call him
our high to be ruled my mastery: nay,
please your lords; prock them our complexions.

Clown:
How man, if mistress, he be it is. Sir John!
Mark, how makes day 'pardon, you both,
you live, sorrow away the writ of half?

KATHAS MOWARD I; that may, thind she not perceive?

GLOUCESTER:
That castle fights slaughter offer thee were with me,
Her must covert her beard's courses of their gore,
Cry want down itself toward to thy death;
So she miseme in stay.

CLARENCE:
As Clarence, thou hast a faw, and seeking me
To more than nature me no carried to the blow.

BALTHASTA:
O men fatal are and Jule?

PARIS:
Ours, I slain, my lord.

GREMIO:
I do please you so:
Brothen, get this is he this deed;
At little the living of my l